In [2]:
# Check Python and package versions before setting up model training.
# This does not install or change anything.

import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for package in ["tensorflow", "keras", "tfswin", "numpy"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.11.17
tensorflow: 2.17.1
keras: 3.4.1
tfswin: 4.1.1
numpy: 1.26.4


In [3]:
# Verify that the notebook uses the training environment
# and that the model libraries import successfully.

import sys
import tensorflow as tf
import keras
import numpy as np
from tfswin import SwinTransformerV2Tiny256
from importlib.metadata import version

print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("NumPy:", np.__version__)
print("tfswin:", version("tfswin"))
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("Imports successful.")

Python executable: /Users/hf/Documents/CassaDx/.venv-train/bin/python
TensorFlow: 2.17.1
Keras: 3.4.1
NumPy: 1.26.4
tfswin: 4.1.1
Available GPUs: []
Imports successful.


In [4]:
# Load pretrained DenseNet-121, freeze its layers, and test
# whether it produces feature maps from a 256 × 256 input.

dense_branch = keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
)

dense_branch.trainable = False

# A dummy image checks the model's operation, not its accuracy.
dummy_image = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
dense_input = keras.applications.densenet.preprocess_input(dummy_image)

dense_features = dense_branch(dense_input, training=False)

print("Feature-map shape:", dense_features.shape)
print("Trainable variables:", len(dense_branch.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(dense_features)).numpy()))

Feature-map shape: (1, 8, 8, 1024)
Trainable variables: 0
All values finite: True


In [5]:
# Load pretrained SwinV2-Tiny, freeze its layers, and test
# its feature output. Built-in preprocessing handles raw 0–255 pixels.

swin_branch = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)

swin_branch.trainable = False

dummy_image = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
swin_features = swin_branch(dummy_image, training=False)

print("Feature-map shape:", swin_features.shape)
print("Trainable variables:", len(swin_branch.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(swin_features)).numpy()))

Feature-map shape: (1, 8, 8, 768)
Trainable variables: 0
All values finite: True


In [6]:
# Combine the frozen DenseNet and SwinV2 branches.
# Train only the new five-class classifier in this initial baseline.

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

# Each branch receives its required preprocessing exactly once.
dense_inputs = keras.applications.densenet.preprocess_input(inputs)
dense_maps = dense_branch(dense_inputs, training=False)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D(
    name="dense_pool"
)(dense_maps)

swin_vector = keras.layers.GlobalAveragePooling2D(
    name="swin_pool"
)(swin_maps)

combined = keras.layers.Concatenate(name="feature_fusion")(
    [dense_vector, swin_vector]
)

outputs = keras.layers.Dense(
    5, activation="softmax", name="disease_classifier"
)(combined)

hybrid_model = keras.Model(inputs, outputs, name="CassaDx_baseline")

# Check the combined model using one dummy image.
predictions = hybrid_model(dummy_image, training=False)

print("Output shape:", predictions.shape)
print("Probability sum:", float(tf.reduce_sum(predictions[0]).numpy()))
print("Trainable parameters:",
      sum(int(np.prod(v.shape)) for v in hybrid_model.trainable_variables))
print("All values finite:",
      bool(tf.reduce_all(tf.math.is_finite(predictions)).numpy()))

Output shape: (1, 5)
Probability sum: 1.0
Trainable parameters: 8965
All values finite: True


In [7]:
# Run one real training image through the hybrid to check the
# complete input path. This does not train or evaluate accuracy.

import csv
import json
from pathlib import Path
from PIL import Image

data_dir = Path("data")
split_dir = Path("splits")

with (split_dir / "train.csv").open(newline="") as file:
    train_records = list(csv.DictReader(file))

with (split_dir / "training_config.json").open() as file:
    training_config = json.load(file)

disease_names = training_config["label_to_disease"]
sample = train_records[0]

with Image.open(data_dir / "train_images" / sample["image_id"]) as img:
    img = img.convert("RGB").resize(
        (256, 256), Image.Resampling.BILINEAR
    )
    image_array = np.array(img, dtype=np.float32)

# Add the batch dimension. Keep raw pixels in the 0–255 range:
# the model handles normalization separately for each branch.
image_batch = image_array[np.newaxis, ...]
probabilities = hybrid_model(image_batch, training=False).numpy()[0]

print("Dataset label:", disease_names[sample["label"]])
print("Input shape:", image_batch.shape)
print("All outputs finite:", bool(np.isfinite(probabilities).all()))
print("Probability sum:", float(probabilities.sum()))

Dataset label: Healthy
Input shape: (1, 256, 256, 3)
All outputs finite: True
Probability sum: 1.0


In [8]:
# Export the current hybrid and attempt a standard FP32 TFLite conversion.
# This checks compatibility before we spend time training.

from pathlib import Path

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

saved_model_dir = export_dir / "hybrid_export_check"
tflite_path = export_dir / "hybrid_export_check_fp32.tflite"

hybrid_model.export(str(saved_model_dir))

converter = tf.lite.TFLiteConverter.from_saved_model(
    str(saved_model_dir)
)

# Require standard TFLite operations for this first compatibility check.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

tflite_model = converter.convert()
tflite_path.write_bytes(tflite_model)

print("\nFP32 conversion successful.")
print("Saved:", tflite_path)
print("File size (MiB):", round(len(tflite_model) / (1024 ** 2), 2))

INFO:tensorflow:Assets written to: models/hybrid_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_export_check/assets


Saved artifact at 'models/hybrid_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5072211216: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067255504: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5067259728: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067258576: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5067259536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067259152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067258960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067257040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5067257808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5068481040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5068484112: TensorSpec(shape

W0000 00:00:1792154809.020823 11798197 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792154809.020860 11798197 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 17:46:49.021487: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_export_check
2026-10-16 17:46:49.041443: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 17:46:49.041460: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_export_check
2026-10-16 17:46:49.259007: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-10-16 17:46:49.293200: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 17:46:50.633555: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_export_check
2026-10-16 17:46:51.046438: I ten

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(callsite(fused["Roll:", "CassaDx_baseline_1/swin2_tiny_256_1/layers.0_1/blocks.0_1/Roll@__inference___call___49330"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___52657"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): 'tf.Roll' op is neither a custom op nor a flex op
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(callsite(fused["Roll:", "CassaDx_baseline_1/swin2_tiny_256_1/layers.0_1/blocks.0_1/Roll@__inference___call___49330"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper___call___52657"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): Error code: ERROR_NEEDS_FLEX_OPS
<unknown>:0: error: loc(callsite(callsite(fused["Transpose:", "CassaDx_baseline_1/swin2_tiny_256_1/layers

In [9]:
# Test FP32 export with TensorFlow fallback operations enabled.
# This is a diagnostic artifact, not a full-INT8 deployment model.

converter = tf.lite.TFLiteConverter.from_saved_model(
    "models/hybrid_export_check"
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

fallback_model = converter.convert()

fallback_path = Path("models/hybrid_export_check_flex_fp32.tflite")
fallback_path.write_bytes(fallback_model)

print("\nFallback FP32 conversion successful.")
print("Saved:", fallback_path)
print("File size (MiB):",
      round(len(fallback_model) / (1024 ** 2), 2))

W0000 00:00:1792154824.588836 11798197 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792154824.588852 11798197 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 17:47:04.589076: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_export_check
2026-10-16 17:47:04.607467: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 17:47:04.607483: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_export_check
2026-10-16 17:47:04.845259: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 17:47:06.146814: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_export_check
2026-10-16 17:47:06.603248: I tensorflow/cc/saved_model/loader.cc:462] SavedModel load for tags { serve }; Status: success: OK. Took 2014172 microseconds.
2026-10-16 


Fallback FP32 conversion successful.
Saved: models/hybrid_export_check_flex_fp32.tflite
File size (MiB): 143.26


In [ ]:
# Compare the exported FP32 model with the original Keras model
# using the same real image. This checks numerical agreement, not accuracy.

interpreter = tf.lite.Interpreter(
    model_path="models/hybrid_export_check_flex_fp32.tflite"
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

test_input = image_batch.astype(np.float32)

keras_output = hybrid_model(test_input, training=False).numpy()

interpreter.set_tensor(input_info["index"], test_input)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_info["index"])

max_difference = np.max(np.abs(keras_output - tflite_output))

print("Keras probabilities:", keras_output[0])
print("TFLite probabilities:", tflite_output[0])
print("Maximum absolute difference:", float(max_difference))
print("Outputs close:",
      np.allclose(keras_output, tflite_output, atol=1e-4, rtol=1e-3))

: 

In [1]:
# Test equivalent window operations using transposes of at most 6 dimensions.
# This cell does not modify the installed library or build the model.

import tensorflow as tf
from keras import ops
from tfswin.window import (
    window_partition,
    window_reverse,
    window_partition_fused,
    window_reverse_fused,
)

def partition_for_export(
    inputs, height, width, window_size, num_heads,
    qkv_mult=3, dtype=None, name=None
):
    windows = window_partition(
        inputs, height, width, window_size, dtype=dtype, name=name
    )
    head_channels = windows.shape[-1] // (qkv_mult * num_heads)

    outputs = ops.reshape(
        windows,
        [-1, window_size ** 2, qkv_mult, num_heads, head_channels]
    )
    return ops.transpose(outputs, [2, 0, 3, 1, 4])


def reverse_for_export(
    inputs, height, width, window_size, num_heads,
    dtype=None, name=None
):
    head_channels = inputs.shape[-1]
    outputs = ops.transpose(inputs, [0, 2, 1, 3])
    outputs = ops.reshape(
        outputs,
        [-1, window_size ** 2, num_heads * head_channels]
    )
    return window_reverse(
        outputs, height, width, window_size, dtype=dtype, name=name
    )


# Compare the original and replacement operations on identical test data.
x = tf.reshape(
    tf.range(32 * 32 * 288, dtype=tf.float32),
    (1, 32, 32, 288)
)

original_partition = window_partition_fused(x, 32, 32, 16, 3)
replacement_partition = partition_for_export(x, 32, 32, 16, 3)
tf.debugging.assert_equal(original_partition, replacement_partition)

test_windows = original_partition[2]
original_reverse = window_reverse_fused(test_windows, 32, 32, 16, 3)
replacement_reverse = reverse_for_export(test_windows, 32, 32, 16, 3)
tf.debugging.assert_equal(original_reverse, replacement_reverse)

print("Partition: exact match.")
print("Reverse: exact match.")

Partition: exact match.
Reverse: exact match.


In [2]:
# Compare the original and modified SwinV2 branch on the same input.
# Both versions use the same pretrained weights and inference settings.

import gc
import numpy as np
import tfswin.window as window_module
import tfswin.winatt as attention_module
from tfswin import SwinTransformerV2Tiny256

test_image = tf.random.stateless_uniform(
    (1, 256, 256, 3),
    seed=(42, 0),
    minval=0,
    maxval=255,
    dtype=tf.float32,
)

original_swin = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)

reference_features = original_swin(
    test_image, training=False
).numpy()

del original_swin
gc.collect()

# Replace both the definitions and the references used by attention.
window_module.window_partition_fused = partition_for_export
window_module.window_reverse_fused = reverse_for_export
attention_module.window_partition_fused = partition_for_export
attention_module.window_reverse_fused = reverse_for_export

swin_branch = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)
swin_branch.trainable = False

updated_features = swin_branch(test_image, training=False).numpy()

print("Feature shape:", updated_features.shape)
print("Maximum difference:",
      float(np.max(np.abs(reference_features - updated_features))))
print("Outputs match:",
      np.allclose(reference_features, updated_features,
                  atol=1e-5, rtol=1e-4))

Feature shape: (1, 8, 8, 768)
Maximum difference: 0.0
Outputs match: True


In [3]:
# Rebuild the hybrid with the modified SwinV2 branch and export it.
# Save a test input and reference output for a separate runtime check.

import keras
from pathlib import Path

keras.utils.set_random_seed(42)

dense_branch = keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
)
dense_branch.trainable = False

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

dense_maps = dense_branch(
    keras.applications.densenet.preprocess_input(inputs),
    training=False,
)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D()(dense_maps)
swin_vector = keras.layers.GlobalAveragePooling2D()(swin_maps)
combined = keras.layers.Concatenate()([dense_vector, swin_vector])
outputs = keras.layers.Dense(
    5, activation="softmax", name="disease_classifier"
)(combined)

hybrid_model = keras.Model(inputs, outputs, name="CassaDx_baseline")

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

np.save(export_dir / "export_test_input.npy", test_image.numpy())
np.save(
    export_dir / "export_test_reference.npy",
    hybrid_model(test_image, training=False).numpy(),
)

saved_path = export_dir / "hybrid_lowrank_export_check"
hybrid_model.export(str(saved_path))

converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_path))
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

converted = converter.convert()
output_path = export_dir / "hybrid_lowrank_flex_fp32.tflite"
output_path.write_bytes(converted)

print("\nModified hybrid exported.")
print("File:", output_path)
print("Size (MiB):", round(len(converted) / (1024 ** 2), 2))

INFO:tensorflow:Assets written to: models/hybrid_lowrank_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_lowrank_export_check/assets


Saved artifact at 'models/hybrid_lowrank_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5139452688: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5133338960: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5128076752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5128076944: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5128075984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128077136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128076176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5128073680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131749712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131750096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5131748944: TensorSp

W0000 00:00:1792156728.135809 11814835 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792156728.136518 11814835 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 18:18:48.138026: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_lowrank_export_check
2026-10-16 18:18:48.157872: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 18:18:48.157890: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_lowrank_export_check
2026-10-16 18:18:48.393093: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-10-16 18:18:48.450282: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 18:18:49.806311: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_lowrank_export_check
2026-10-1


Modified hybrid exported.
File: models/hybrid_lowrank_flex_fp32.tflite
Size (MiB): 143.27


In [4]:
# Test a replacement for Roll that selects values in their shifted order.
# This checks the replacement without changing the model.

import tensorflow as tf
import numpy as np

def roll_for_export(inputs, shifts, axes):
    outputs = inputs

    for shift, axis in zip(shifts, axes):
        length = tf.shape(outputs)[axis]
        shift = tf.cast(shift, tf.int32)

        indices = tf.math.floormod(
            tf.range(length) - shift,
            length,
        )
        outputs = tf.gather(outputs, indices, axis=axis)

    return outputs


# Check forward shifts, reverse shifts, and no shift.
sample = tf.reshape(
    tf.range(2 * 8 * 8 * 3, dtype=tf.float32),
    (2, 8, 8, 3),
)

for shift in [-4, 0, 4]:
    expected = tf.roll(sample, [shift, shift], [1, 2])
    actual = roll_for_export(sample, [shift, shift], [1, 2])

    np.testing.assert_array_equal(expected.numpy(), actual.numpy())
    print(f"Shift {shift}: exact match.")

Shift -4: exact match.
Shift 0: exact match.
Shift 4: exact match.


In [5]:
# Apply the Roll replacement only inside Swin's module.
# Compare the complete branch before and after the change.

import tfswin.swin as swin_module
from tfswin import SwinTransformerV2Tiny256

# Save the current branch's output before applying the replacement.
reference_features = swin_branch(
    test_image, training=False
).numpy()

# Preserve all other operations and replace only Roll.
class SwinExportOps:
    def __init__(self, original_ops):
        self.original_ops = original_ops

    def __getattr__(self, name):
        return getattr(self.original_ops, name)

    def roll(self, inputs, shift, axis):
        return roll_for_export(inputs, shift, axis)


swin_module.ops = SwinExportOps(swin_module.ops)

# Build a new branch so the replacement is included in its graph.
export_swin = SwinTransformerV2Tiny256(
    weights="imagenet",
    include_top=False,
    input_shape=(256, 256, 3),
    pooling=None,
    include_preprocessing=True,
)
export_swin.trainable = False

updated_features = export_swin(
    test_image, training=False
).numpy()

print("Feature shape:", updated_features.shape)
print(
    "Maximum difference:",
    float(np.max(np.abs(reference_features - updated_features))),
)

np.testing.assert_allclose(
    reference_features,
    updated_features,
    atol=1e-5,
    rtol=1e-5,
)

print("Check passed: Swin features match.")

# Use the checked branch for the next export.
swin_branch = export_swin

Feature shape: (1, 8, 8, 768)
Maximum difference: 0.0
Check passed: Swin features match.


In [6]:
# Rebuild the hybrid with the checked Swin branch, keeping its classifier.
# Verify matching predictions, then attempt export without Flex.

import keras
from pathlib import Path

reference = hybrid_model(test_image, training=False).numpy()

inputs = keras.Input(shape=(256, 256, 3), name="leaf_image")

dense_maps = dense_branch(
    keras.applications.densenet.preprocess_input(inputs),
    training=False,
)
swin_maps = swin_branch(inputs, training=False)

dense_vector = keras.layers.GlobalAveragePooling2D()(dense_maps)
swin_vector = keras.layers.GlobalAveragePooling2D()(swin_maps)

combined = keras.layers.Concatenate()([dense_vector, swin_vector])

# Reuse the existing classifier rather than creating new random weights.
classifier = hybrid_model.get_layer("disease_classifier")
outputs = classifier(combined)

export_model = keras.Model(
    inputs, outputs, name="CassaDx_export"
)

updated = export_model(test_image, training=False).numpy()

np.testing.assert_allclose(
    reference, updated, atol=1e-5, rtol=1e-5
)
print("Hybrid predictions match.")

# Save matching input/output files for the separate runtime test.
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

np.save(model_dir / "builtin_test_input.npy", test_image.numpy())
np.save(model_dir / "builtin_test_reference.npy", updated)

saved_path = model_dir / "hybrid_builtin_export_check"
export_model.export(str(saved_path))

# Allow standard TFLite operations only.
converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_path))
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

converted = converter.convert()

output_path = model_dir / "hybrid_builtin_fp32.tflite"
output_path.write_bytes(converted)

print("Export without Flex succeeded.")
print("Saved:", output_path)
print("Size (MiB):", round(len(converted) / (1024 ** 2), 2))

Hybrid predictions match.
INFO:tensorflow:Assets written to: models/hybrid_builtin_export_check/assets


INFO:tensorflow:Assets written to: models/hybrid_builtin_export_check/assets


Saved artifact at 'models/hybrid_builtin_export_check'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='leaf_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  5185534672: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5185533328: TensorSpec(shape=(3,), dtype=tf.float32, name=None)
  5168732432: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5168728208: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5168727248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168728016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168731856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168728784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168729936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168731280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5168733008: TensorSp

W0000 00:00:1792158357.246150 11814835 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1792158357.246519 11814835 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2026-10-16 18:45:57.248180: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: models/hybrid_builtin_export_check
2026-10-16 18:45:57.267687: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-10-16 18:45:57.267713: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: models/hybrid_builtin_export_check
2026-10-16 18:45:57.535020: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-10-16 18:45:58.888683: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: models/hybrid_builtin_export_check
2026-10-16 18:45:59.308092: I tensorflow/cc/saved_model/loader.cc:462] SavedModel load for tags { serve }; Status: success: OK. Took 2059913 m

Export without Flex succeeded.
Saved: models/hybrid_builtin_fp32.tflite
Size (MiB): 143.27


In [1]:
# Rebuild the frozen hybrid with both compatibility fixes applied.
from model_setup import build_hybrid_model
from math import prod

hybrid_model = build_hybrid_model()

print("Output shape:", hybrid_model.output_shape)
print("Trainable parameters:", sum(
    prod(variable.shape)
    for variable in hybrid_model.trainable_variables
))

Output shape: (None, 5)
Trainable parameters: 8965
